In [ ]:
# Multi-course RAG ingestion pipeline using Vision & Caching (Google Colab GPU)
# 1. Install required libraries for SentenceTransformers, ChromaDB, PyMuPDF, BM25, and Gemini API
!pip install -q sentence-transformers chromadb==0.5.9 pymupdf rank_bm25 tqdm google-genai torch pillow

In [ ]:
# 2. Mount Google Drive for persistent storage access
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted successfully.')

In [ ]:
# 3. Load Gemini API Key from Google Colab Secrets
import os
from google.colab import userdata

try:
    os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
    print("Gemini API key loaded successfully.")
except Exception as e:
    print("Error loading API key:", e)

In [ ]:
# 4. Global configuration and hyperparameters (models, chunking, paths, courses)
import torch

# Dense embedding and multimodal vision model names
EMBEDDING_MODEL_NAME = 'intfloat/multilingual-e5-large-instruct'
GEMINI_VISION_MODEL  = 'gemini-2.5-flash'

# Filtering thresholds to discard tiny text fragments and small/decorative images
MIN_CHUNK_WORDS  = 15
MIN_IMAGE_WIDTH  = 500
MIN_IMAGE_HEIGHT = 250

# ChromaDB vector store directory
CHROMA_PATH = "chroma_db"

# Multi-course configuration dictionary and Google Drive directory paths
COURSES = {
    "databases": {
        "name": "Database Systems",
        "collection": "db_course",
        "bm25_index": "bm25_databases.pkl",
        "folder": "/content/drive/MyDrive/databases"
    },
    "parallel-distributed": {
        "name": "Parallel and Distributed Computing",
        "collection": "parallel_course",
        "bm25_index": "bm25_parallel.pkl",
        "folder": "/content/drive/MyDrive/parallel-distributed"
    }
}

# Verify GPU device availability
if torch.cuda.is_available():
    print(f"GPU Available: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: GPU not detected. Switching to a GPU runtime is recommended.")

In [ ]:
# 5. Greek text tokenizer for BM25 sparse retrieval
import string
import unicodedata
from typing import List

def tokenize_greek(text: str) -> List[str]:
    """Strips accents, converts to lowercase, and removes punctuation for Greek BM25 index terms.

    Args:
        text: Input string to tokenize.

    Returns:
        List of cleaned token terms.
    """
    text = text.lower()
    text = ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')
    for char in ['-', '«', '»', '“', '”', '·', ':', ';', '!', '?', ',', '.', '(', ')', '[', ']', '{', '}']:
        text = text.replace(char, ' ')
    text = text.translate(str.maketrans('', '', string.punctuation))
    return [t for t in text.split() if len(t) > 1]

print("Greek tokenizer loaded.")

In [ ]:
# 6. Persistent Vision LLM image description cache management
import os
import json

# Cache file on Google Drive to avoid redundant Gemini API requests
CACHE_FILE = "/content/drive/MyDrive/Thesis/vision_cache.json"

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        vision_cache = json.load(f)
    print(f"Loaded {len(vision_cache)} cached image descriptions.")
else:
    vision_cache = {}
    os.makedirs(os.path.dirname(CACHE_FILE), exist_ok=True)
    print("Created new cache file.")

def save_cache() -> None:
    """Persists current image description cache dictionary to Google Drive."""
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(vision_cache, f, ensure_ascii=False, indent=2)

In [ ]:
# 7. Extract detailed image descriptions using Gemini Vision API with rate limit retries
import hashlib
import time
import sys
import io
from typing import Tuple
from google import genai
from PIL import Image

vision_client = genai.Client()

def describe_image(image_bytes: bytes, image_ext: str, page_num: int, surrounding_text: str) -> Tuple[str, bool]:
    """Generates image description using Gemini API with caching and rate limit retry logic.

    Args:
        image_bytes: Raw binary image payload.
        image_ext: Image format extension string.
        page_num: Page number location.
        surrounding_text: Text surrounding the image for context.

    Returns:
        Tuple of (description_string, is_cached_boolean).
    """
    img_hash = hashlib.md5(image_bytes).hexdigest()

    # Check cache for deduplication
    if img_hash in vision_cache:
        return vision_cache[img_hash], True

    # Limit surrounding text to 100 words to constrain prompt size
    context_hint = " ".join(surrounding_text.split()[:100]) if surrounding_text else ""
    prompt = (
        "Περίγραψε αυτή την εικόνα περιεκτικά στα Ελληνικά. "
        f"Το κείμενο της σελίδας γύρω από την εικόνα είναι: {context_hint}. "
        "Αν είναι διάγραμμα: γράψε τα components και τη ροή. Αν είναι πίνακας: κάνε transcribe όλα τα δεδομένα. Αν είναι κώδικας: αντέγραψέ τον ακριβώς. "
        "ΣΗΜΑΝΤΙΚΟ: Αν η εικόνα είναι απλά ένα εταιρικό/πανεπιστημιακό λογότυπο (logo), ένα εικονίδιο, διακοσμητικό στοιχείο ή υδατογράφημα χωρίς καμία απολύτως εκπαιδευτική αξία για το μάθημα, απάντησε ΑΥΣΤΗΡΑ ΚΑΙ ΜΟΝΟ με τη λέξη 'SKIP'."
    )

    max_retries = 3
    for attempt in range(max_retries):
        try:
            img = Image.open(io.BytesIO(image_bytes))
            response = vision_client.models.generate_content(
                model=GEMINI_VISION_MODEL,
                contents=[img, prompt]
            )
            result = response.text.strip()

            # Update persistent cache on successful response
            vision_cache[img_hash] = result
            save_cache()

            return result, False

        except Exception as e:
            error_msg = str(e)
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg or "503" in error_msg:
                wait_time = 65
                print(f"
  [Rate Limit] Page {page_num}: Waiting {wait_time}s... (Attempt {attempt+1}/{max_retries})")
                time.sleep(wait_time)
            else:
                print(f"
  Vision API Error (Page {page_num}): {e}")
                return "", False

    print(f"
  Vision processing failed after {max_retries} attempts.")
    sys.exit("ABORT: Exceeded available Gemini API quotas.")

In [ ]:
# 8. Document processing and text/image chunking (PDF & TXT)
import fitz
import glob
from typing import List, Dict, Any, Tuple

def process_pdf(pdf_path: str, current_chunk_id: int) -> Tuple[List[Dict[str, Any]], int]:
    """Extracts text and image descriptions from PDF document and generates structured chunks.

    Args:
        pdf_path: File system path to target PDF document.
        current_chunk_id: Incremental chunk ID counter.

    Returns:
        Tuple containing list of chunk dictionaries and updated chunk ID counter.
    """
    filename_base = os.path.basename(pdf_path)
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    page_texts = []

    for page_num in range(total_pages):
        page = doc.load_page(page_num)
        text = page.get_text("text").strip()

        # Extract and describe embedded page images
        image_list = page.get_images(full=True)
        for image_index, img in enumerate(image_list):
            try:
                xref = img[0]
                image_data = doc.extract_image(xref)

                # Filter out small graphics and logos below dimension or area thresholds
                w = image_data.get("width",  0)
                h = image_data.get("height", 0)
                if w < MIN_IMAGE_WIDTH or h < MIN_IMAGE_HEIGHT or (w * h < 150000):
                    continue

                image_bytes = image_data["image"]
                image_ext   = image_data.get("ext", "png")

                vision_text, is_cached = describe_image(image_bytes, image_ext, page_num + 1, text)

                if vision_text.upper() == "SKIP" or len(vision_text.strip()) < 20:
                    continue

                if not is_cached:
                    time.sleep(10)

                text += f"
[VISION_EXTRACT - Image {image_index+1}]: {vision_text}"

            except Exception:
                continue

        if text:
            page_texts.append((page_num, text))

    doc.close()

    # Aggregate text blocks into chunks meeting minimum word count
    pdf_chunks = []
    buffer_text = ""
    buffer_start_page = 0
    for page_num, text in page_texts:
        if not buffer_text:
            buffer_text = text
            buffer_start_page = page_num
        elif len(buffer_text.split()) < MIN_CHUNK_WORDS:
            buffer_text += " " + text
        else:
            pdf_chunks.append({
                "text": buffer_text,
                "metadata": {"source": filename_base, "page": buffer_start_page, "type": "pdf"},
                "id": f"chunk_{current_chunk_id}"
            })
            current_chunk_id += 1
            buffer_text = text
            buffer_start_page = page_num

    if buffer_text and len(buffer_text.split()) >= MIN_CHUNK_WORDS:
        pdf_chunks.append({
            "text": buffer_text,
            "metadata": {"source": filename_base, "page": buffer_start_page, "type": "pdf"},
            "id": f"chunk_{current_chunk_id}"
        })
        current_chunk_id += 1

    return pdf_chunks, current_chunk_id

def process_txt(txt_path: str, current_chunk_id: int) -> Tuple[List[Dict[str, Any]], int]:
    """Processes plain text document into structured chunks based on paragraph blocks.

    Args:
        txt_path: Path to plain text file.
        current_chunk_id: Incremental chunk ID counter.

    Returns:
        Tuple containing list of chunk dictionaries and updated chunk ID counter.
    """
    filename_base = os.path.basename(txt_path)
    with open(txt_path, "r", encoding="utf-8") as f:
        content = f.read()

    blocks = [b.strip() for b in content.split('

') if b.strip()]
    txt_chunks = []
    for block in blocks:
        if len(block.split()) < MIN_CHUNK_WORDS:
            continue
        txt_chunks.append({
            "text": block,
            "metadata": {"source": filename_base, "type": "txt"},
            "id": f"chunk_{current_chunk_id}"
        })
        current_chunk_id += 1

    return txt_chunks, current_chunk_id

In [ ]:
# 9. Course ingestion pipeline: chunking, GPU embeddings, ChromaDB & BM25 indexing
import pickle
import chromadb
from tqdm.notebook import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

def ingest_course(course_key: str) -> None:
    """Executes complete document processing pipeline: chunking, GPU embeddings, ChromaDB storage & BM25 index creation.

    Args:
        course_key: Identifier key corresponding to an entry in the COURSES dictionary.
    """
    course_info = COURSES[course_key]
    folder_path = course_info["folder"]

    if not os.path.exists(folder_path):
        print(f"Error: Directory {folder_path} not found!")
        return

    pdf_files = glob.glob(os.path.join(folder_path, "*.pdf"))
    txt_files = glob.glob(os.path.join(folder_path, "*.txt"))

    if not pdf_files and not txt_files:
        print(f"No PDF or TXT files found in directory {folder_path}")
        return

    print(f"
=== Starting Ingestion: {course_info['name']} ===")
    print(f"Found {len(pdf_files)} PDF and {len(txt_files)} TXT files.")

    all_course_chunks = []
    current_chunk_id = 0

    if pdf_files:
        for pdf_path in tqdm(pdf_files, desc="Processing PDF"):
            chunks, current_chunk_id = process_pdf(pdf_path, current_chunk_id)
            all_course_chunks.extend(chunks)

    if txt_files:
        for txt_path in tqdm(txt_files, desc="Processing TXT"):
            chunks, current_chunk_id = process_txt(txt_path, current_chunk_id)
            all_course_chunks.extend(chunks)

    if not all_course_chunks:
        print("No valid chunks were generated for this course.")
        return

    texts = [c["text"] for c in all_course_chunks]
    metadata = [c["metadata"] for c in all_course_chunks]
    ids = [c["id"] for c in all_course_chunks]

    # Generate dense vector embeddings on GPU
    print(f"
Loading Embedding model: {EMBEDDING_MODEL_NAME}...")
    embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    print(f"Generating embeddings for {len(texts)} chunks on GPU...")
    embeddings = embed_model.encode(texts, show_progress_bar=True, batch_size=64).tolist()

    # Store embeddings and metadata in ChromaDB vector store
    print("Saving to ChromaDB...")
    chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
    collection_name = course_info["collection"]

    try:
        chroma_client.delete_collection(name=collection_name)
        print(f"Deleted existing collection: {collection_name}")
    except Exception:
        pass

    collection = chroma_client.get_or_create_collection(name=collection_name)

    BATCH_SIZE = 5000
    for i in range(0, len(texts), BATCH_SIZE):
        end_idx = min(i + BATCH_SIZE, len(texts))
        collection.add(
            documents=texts[i:end_idx],
            embeddings=embeddings[i:end_idx],
            metadatas=metadata[i:end_idx],
            ids=ids[i:end_idx]
        )
    print(f"Stored {len(all_course_chunks)} chunks in ChromaDB collection ({collection_name}).")

    # Build and serialize BM25 index for keyword search
    print("Building BM25 Index...")
    tokenized_corpus = [tokenize_greek(t) for t in texts]
    bm25 = BM25Okapi(tokenized_corpus)
    bm25_filename = course_info["bm25_index"]

    if os.path.exists(bm25_filename):
        os.remove(bm25_filename)

    with open(bm25_filename, "wb") as f:
        pickle.dump({"bm25": bm25, "chunks": texts, "metadata": metadata, "ids": ids}, f)
    print(f"Saved BM25 index to {bm25_filename}.")

    # Release client resources and trigger garbage collection
    del collection
    del chroma_client
    import gc
    gc.collect()
    print("ChromaDB connection closed successfully.")

In [ ]:
# 10. Execute ingestion for selected course or all courses
if not os.environ.get("GEMINI_API_KEY"):
    print("ERROR: GEMINI_API_KEY is not defined in Colab Secrets.")
else:
    print("=== Available Courses ===")
    for key, info in COURSES.items():
        print(f"  [{key}] -> {info['name']}")

    choice = input("
Select course (databases / parallel-distributed / all): ").strip()

    if choice == "all":
        for key in COURSES:
            ingest_course(key)
    elif choice in COURSES:
        ingest_course(choice)
    else:
        print("Invalid selection.")

In [ ]:
# 11. Compress ChromaDB vector store and download index files (.zip & .pkl)
import shutil
from google.colab import files

print("=== Archiving & Downloading Output Files ===")

# Clean temporary lock files and compress ChromaDB directory
if os.path.exists(CHROMA_PATH):
    for root, dirs, files_list in os.walk(CHROMA_PATH):
        for f in files_list:
            if f.endswith(('-wal', '-shm')):
                try:
                    os.remove(os.path.join(root, f))
                except Exception:
                    pass

    shutil.make_archive("chroma_db", "zip", CHROMA_PATH)
    print("Created chroma_db.zip")
else:
    print("ChromaDB directory not found.")

# Download zip archive and BM25 index pickle files
print("
Initiating file downloads...")
if os.path.exists("chroma_db.zip"):
    files.download("chroma_db.zip")

for key in COURSES:
    bm25_file = COURSES[key]["bm25_index"]
    if os.path.exists(bm25_file):
        files.download(bm25_file)

print("
Processing completed successfully!")